In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
from IPython.display import clear_output

%pip install kagglehub catboost lightgbm tqdm -q

clear_output()

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import os
from tqdm import tqdm
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub

from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

import warnings
warnings.filterwarnings('ignore')



%matplotlib inline


# Task 1: Write your code here: Read the dataset Q1_data.csv using read_csv()
csv_path = os.path.join(path, "Q1_data.csv")

df = pd.read_csv(csv_path)


In [ ]:
# Task 2: Write your code here: Inspect the first few rows using head()
df.head()

In [ ]:
# Task 3: Write your code here: Display dataset information using info()
df.info()

In [ ]:
# Task 4: Write your code here: Show statistical description using describe()
df.describe()


In [ ]:
# Task 5: Write your code here: Plot the target distribution (delivery_time) Your target is the column: "delivery_time"
# 1. What does our target variable (charges) look like?
def check_target_distribution(df, target_column):
  df[target_column].hist(bins=30, edgecolor='black')

  plt.title(f"Target Distribution ({target_column})")
  plt.xlabel(target_column)
  plt.ylabel("Frequency")
  plt.grid(False)

  plt.show()

check_target_distribution(df, "Delivery_Time")

In [ ]:
# Task 1: Write your code here: Drop the 'Order_ID' column from the data
df = df.drop(columns="Order_ID", axis=1)


In [ ]:
# Task 2: Write your code here: Handle missing values appropriately (Hint: I guess you want to have a closer look at the columns with missing values :) )
# Analyze missing values
missing_percentage = (df.isnull().sum() / len(df)) * 100
missing_data = pd.DataFrame({
    'Column': missing_percentage.index,
    'Missing_Percentage': missing_percentage.values
})
missing_data = missing_data[missing_data['Missing_Percentage'] > 0].sort_values('Missing_Percentage', ascending=False)

print("Missing Data Analysis:")
missing_data.head() # -_-
print("---------")
# 2. Do we have missing values? our entries are 1663

def check_missing_values(df):
  missing_values = df.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")

check_missing_values(df)


In [ ]:
# cont. task 2: (Hint: I guess you want to have a closer look at the columns with missing values :) )
# our entries are 1663
# Select relevant columns, we do not select 'model' (too many unique values, too sparse and will hurt the model performance)
cols = ['Distance_km',	'Weather',	'Traffic_Level',	'Time_of_Day',	'Vehicle_Type',	'Preparation_Time_min',	'Courier_Experience_yrs',	'Delivery_Time']
df_clean = df[cols].copy() # important to not miss with the reprodasibility

# Drop rows where target (Delivery Time) or key features are missing - can't predict without them

# missing values are small compared to the data entries, can i just drop all? it seems all are important for predicting, when we have a rainy day the roads are slippry hence
# takes more time, also the traffic is universly agreed on I think, time of day .. well for that i can only link it to traffic -_-, the experience ..
# its likned to knowing shortcuts and what not though is it really that important? just a bit maybe?. hmm for weather, wanted to argue we mostly have sunny dry but no not sure where the data was from
# hmmm will proceed with the experience and time of day to be moved into filling maybe?

print(f"Before: {df_clean.shape}")
df_clean = df_clean.dropna(subset=['Delivery_Time','Weather','Traffic_Level'])
print(f"After dropping missing values: {df_clean.shape}")

# mode is most representative, we have the experience and time of day maybe? the mode is the most occured .. maybe is fine
df_clean['Time_of_Day'] = df_clean['Time_of_Day'].fillna(df_clean['Time_of_Day'].mode()[0])
df_clean['Courier_Experience_yrs'] = df_clean['Courier_Experience_yrs'].fillna(df_clean['Courier_Experience_yrs'].mode()[0])


print("Missing values remaining:", df_clean.isnull().sum().sum())

In [ ]:
# Task 3: Write your code here: Check and remove duplicates if any exist
# 4. Do we have duplicate samples?
def check_duplicates(df_clean):
  duplicates = df_clean.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df_clean.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df_clean)

In [ ]:
# Task 4: Write your code here: Encode categorical variables if needed (Bonus if used One Hot Encoding)
# 3. Do we have categorical columns?
categorical_cols = df_clean.select_dtypes(include=["object"]).columns

print("Categorical Columns:", list(categorical_cols))

In [ ]:
# Encode categorical columns - converts text to integers
categorical_cols = ['Weather', 'Traffic_Level', 'Time_of_Day', 'Vehicle_Type']
for col in categorical_cols:
    le = LabelEncoder()
    df_clean[col] = le.fit_transform(df_clean[col].astype(str))

df_clean.head()


In [ ]:
# Task 5: Write your code here: Apply feature scaling for all features (Use StandardScaler)
# Scale features - fit on train, transform both
from sklearn.preprocessing import StandardScaler

features = df_clean.columns.drop("Delivery_Time")  # DON'T SCALE THE TARGET

scaler = StandardScaler()
df_clean[features] = scaler.fit_transform(df_clean[features])
df_clean.head()

In [ ]:
# Task 6: Write your code here: Check for target imbalance and state if it is imbalanced or not (keep this cell empty if not needed)
# 1. Is the target imbalanced?
def check_target_imbalance(df_clean, target_column):
  print("Target Distribution:")
  print(df_clean[target_column].value_counts(normalize=True))
  sns.countplot(x=df_clean[target_column])
  plt.title("Target Distribution")
  plt.show()

check_target_imbalance(df_clean,"Delivery_Time") # its imbalanced .. ? , in general its safe to use StratifiedKFold since it will act as k-fold if not

In [ ]:
# Task 1: Write your code here: Split the dataset into features (X) and target (y)
X = df_clean.drop("Delivery_Time", axis=1).astype(float)
y = df_clean['Delivery_Time'].astype(float)

In [ ]:
# Task 2,3,4,5: Write your code here: Use the correct split: KFold OR StratifiedKFold
# Train a RandomForest model
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_squared_error as sklearn_mse

models = {"Random Forest Regressor": RandomForestRegressor(n_estimators=200)}
# Storage for results
all_results = {}
y_predction=[]
for name in models:
  all_results[name] = {'mse': []} # Evaluate using MAE (Mean Absolute Error) ONLY

n_splits = 5 # K=5 Folds

# Stratified 5-Fold Cross-Validation, shuffled
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)): #?
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  for model_name, model in models.items():
    print(f"Training {model_name}...")

    # Train
    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_test)
    y_predction.append(y_pred)

    # Calculate metrics
    mse = sklearn_mse(y_test, y_pred)

    # Store results
    all_results[model_name]["mse"].append(mse)



# Print the averaged score across all folds

for model_name in all_results:
  print(f"\n{model_name}:")
  print(f"  MSE:  {np.mean(all_results[model_name]['mse']):.4f}")

"""
average_losses = np.mean(lr_losses, axis=0)

plt.figure(figsize=(10, 6))
plt.plot(average_losses, label='Average Loss')
plt.xlabel('Iteration')
plt.ylabel('MSE Loss')
plt.title('Linear Regression Training Loss (Averaged Across Folds)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()
"""

In [ ]:
# Task 1: Write your code here: Plot feature importance from your trained model
# Feature importance
feature_importance = pd.DataFrame({
    'feature': features,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here: Plot predicted delivery time histogram
def prediction_hist(df_clean, y_predction):
  # here I am suppose to use the y_predicted times I tried to save it into a list but not sure
  print("Presicted Delivery Time dist.:")
  sns.countplot(y_predction) # here i should specify that the predicted values goes to x axis, it shows an error might be due to list?
  plt.title("Predction")
  plt.show()

prediction_hist(df_clean, y_predction)
# ok clearly needs fixing, but hope the idea is ok?

In [ ]:
# Task Bonus: Write your code here: